# Demo: ETL, datasets built on datasets

A **gamey** is a list of integers. A **binning** is fit over a set of them. A **trial** places one
gamey into a binning, and its result is a podium: the three fullest bins. **Metrics** describe a
trial. Each is a dat built from the one below, so the chain is
gamey → binning → trial → metrics, and each seal records the stage below it as a dependency.
(The stage is a trial, not a run: in dvc-dat a run is the sub-dat an `execute` makes inside a dat.)

The kinds are `demo_kinds/etl.py`; the dat mechanics (seal, hash, dependencies) are
`demo-dats`'s. This is the bare bones: each stage is made by hand with its kind's `make`, and
the game set and its version are rows on the ledger (`demo-ledger`).

In [1]:
# a fresh ledger and dat store in throwaway folders, and the four kinds
import os, tempfile
from pathlib import Path

from dvc_dat import Dat, DatManager
from dvc_dat.ledger import Ledger, Ref
from demo_kinds.etl import Gamey, Binning, Trial, Metrics

ledger_path = Path(tempfile.mkdtemp(prefix="demo-etl-"))
os.environ["DAT_LEDGER"] = str(ledger_path)
L = Ledger.open()
dat_store_path = Path(tempfile.mkdtemp(prefix="demo-etl-dats-"))
Dat.manager = DatManager(dat_folders=[str(dat_store_path / "dats")], art_folder=str(dat_store_path / "art"))

## Gameys

In [2]:
# three gameys from G numbers, reproducible from their seeds
for g in ("G1", "G2", "G3"):
    Gamey.make(f"gamey/{g}", n=200, seed=int(g[1:]))
g3 = Dat.manager.load("gamey/G3")
print(f"{len(g3.values())=}  first ten: {g3.values()[:10]}")
print(f"same seed, same list: {Gamey.make('gamey/G3-again', n=200, seed=3).values() == g3.values()}")

len(g3.values())=200  first ten: [267, 602, 630, 625, 379, 645, 521, 369, 678, 557]
same seed, same list: True


## The normie game set

In [3]:
# the normie game set: assertions on the ledger, as NORM is made
L.add("gameset/normie", "contains", Ref("gamey/G1"), by="dan")
L.add("gameset/normie", "contains", Ref("gamey/G2"), by="dan")
L.add("gameset/normie", "contains", Ref("gamey/G3"), by="dan")
normie = [r.name for r in L.objects("gameset/normie", "contains")]
print(f"{normie=}")

normie=['gamey/G1', 'gamey/G2', 'gamey/G3']


In [4]:
# a version on the set: a bump is an assign
L.assign("gameset/normie", "version", "0.1.0", by="dan")
print(f"{L.last('gameset/normie', 'version')=}")

L.last('gameset/normie', 'version')='0.1.0'


## A binning over the set

In [5]:
# the binning for normie 0.1.0, fit over its games (bins and trim at defaults)
b = Binning.make("binning/normie-0.1.0", normie)
print(f"{b.kwargs()=}")
print(f"edges={[round(e, 1) for e in b.edges()]}")

b.kwargs()={'games': ['gamey/G1', 'gamey/G2', 'gamey/G3'], 'bins': 8, 'trim': 0.05}
edges=[220.0, 280.2, 340.5, 400.8, 461.0, 521.2, 581.5, 641.8, 702.0]


In [6]:
# another binning, same set, other parameters: made the same way, named by its parameters
b16 = Binning.make("binning/normie-0.1.0-bins16-trim0", normie, bins=16, trim=0.0)
print(f"{len(b16.edges())=}  {b16.edges()[0]=}  {b16.edges()[-1]=}")

len(b16.edges())=17  b16.edges()[0]=104.0  b16.edges()[-1]=802.0


Trim drops the tails before the edges are drawn, so the default binning spans less than the
untrimmed one and its bins are narrower.

## A trial and its metrics

In [7]:
# a trial: G3 placed into the binning; the podium is the three fullest bins
r = Trial.make("trial/normie-0.1.0-G3", "gamey/G3", b.get_path_name())
print(f"podium={r.podium()}")
print(r.table("placements").head(5))

podium=[2, 7, 1]
   pos  value  bin  residual
0    0    267    0  0.780083
1    1    602    6  0.340249
2    2    630    6  0.804979
3    3    625    6  0.721992
4    4    379    2  0.639004


In [8]:
# metrics over the trial, a dat of its own: values max / min / mean, and the residual inside each bin
m = Metrics.make("metrics/normie-0.1.0-G3", r.get_path_name())
res = m.get_results()
print(f"max={res['max']}  min={res['min']}  mean={res['mean']:.1f}")
print(f"residual: min={res['residual']['min']:.3f}  mean={res['residual']['mean']:.3f}  spread={res['residual']['spread']:.3f}")

max=768  min=108  mean=464.5
residual: min=0.000  mean=0.515  spread=0.321


## The chain, read back

In [9]:
# each seal lists the stage below it; the seal keeps the roots
for d in (m, r, b):
    print(f"{d.get_path_name():30} depends on {sorted(Dat.get(d.get_results(), 'dat.dependencies'))}")

metrics/normie-0.1.0-G3        depends on ['$MANUAL', 'trial/normie-0.1.0-G3']
trial/normie-0.1.0-G3          depends on ['binning/normie-0.1.0']
binning/normie-0.1.0           depends on ['gamey/G1', 'gamey/G2', 'gamey/G3']


The trial loaded both G3 and the binning, but G3 is already below the binning (normie holds it),
so the seal keeps only the binning: the chain is followed metrics → trial → binning → G1, G2, G3.
`$MANUAL` marks a dat sealed by hand rather than by a run; it is trimmed wherever an input
already carries it.

In [10]:
# clean up
import shutil
L.close(); Ledger.default = None
del os.environ["DAT_LEDGER"]
Dat.manager = None
for p in (dat_store_path, ledger_path):
    shutil.rmtree(p)